# RoPE 算子开发实验

按顺序运行下面的代码，需要修改的参数已在代码中标出。


<a id="s-02-02_rope_simd_simt"></a>


## 6.6 RoPE：SIMD 与 SIMT

### 小节概述

目标：读懂 pair-planar 数据布局，编译并运行 A3 SIMD Kernel，再说明 950 SIMT 模板的并行映射。

<img src="images/rope_pair_planar_flow.svg" width="760" style="display:block; margin-left:0;" />

### 为什么要旋转一对通道

RoPE 为 Query、Key 加入位置信息，把相邻的 even/odd 两个通道看作平面中的一个点。位置和通道编号决定旋转角度，cos/sin 是对应角度的余弦和正弦。

例如 `(even,odd)=(1,0)`，旋转 90° 后变成 `(0,1)`。position=0 时角度为 0，cos=1、sin=0，所以输出等于输入。

原数组按 `[e0,o0,e1,o1]` 交错存放；Host 将它拆成 `[e0,e1]` 与 `[o0,o1]` 两段连续数组，方便向量指令批量处理。这就是本例的 pair-planar 布局。


### SIMD Kernel

`GlobalTensor` 对应设备大内存（GM）中的数据；本例的 `LocalTensor` 对应 Vector Core 片上缓冲（UB）中的数据。`TPipe` 管理缓冲，`TQue` 连接 CopyIn → Compute → CopyOut 三个阶段，`BUFFER_NUM` 控制每个队列的缓冲数量。

在 `src/rope_simd_kernel.cpp` 中，`InitBuffer` 先准备队列缓冲；CopyIn 用 `AllocTensor` 取一块缓冲，`DataCopy` 搬入数据，再用 `EnQue` 入队。Compute 用 `DeQue` 取出数据，完成计算后释放输入缓冲，输出经队列交给 CopyOut。

下面摘出 even/odd/cos/sin 的搬入和 `Mul/Sub/Add` 旋转计算，完整流程见该源码。


```cpp
AscendC::DataCopy(xe, xEvenGm, TOTAL_PAIRS);
AscendC::DataCopy(xo, xOddGm, TOTAL_PAIRS);
AscendC::DataCopy(c,  cosGm,  TOTAL_PAIRS);
AscendC::DataCopy(s,  sinGm,  TOTAL_PAIRS);

AscendC::Mul(te1, xe, c, TOTAL_PAIRS);
AscendC::Mul(te2, xo, s, TOTAL_PAIRS);
AscendC::Sub(oe, te1, te2, TOTAL_PAIRS);
AscendC::Mul(to1, xe, s, TOTAL_PAIRS);
AscendC::Mul(to2, xo, c, TOTAL_PAIRS);
AscendC::Add(oo, to1, to2, TOTAL_PAIRS);
```

入口中的 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY)` 把任务限定为 Vector Core。`TOTAL_PAIRS=384` 对应 12 heads × 32 pairs；本实现是教学用单 tile、单缓冲，不预设为性能最优版本。

### RTC Host 与执行

Host 侧生命周期为：创建 RTC Program → 编译 → 读取二进制 → 加载 Vector Core Binary → 获取函数 → 追加 6 个设备指针 → 启动 → 同步 → 校验 → 逆序释放。

```cpp
aclrtcCompileProg(prog, numOptions, options);
aclrtBinaryLoadFromData(elfBin.data(), elfSize, &loadOption, &binHandle);
aclrtBinaryGetFunction(binHandle, "rope_simd", &funcHandle);
aclrtLaunchKernelWithConfig(funcHandle, blockDim, stream, nullptr, argsHandle, nullptr);
aclrtSynchronizeStream(stream);
```

下面编译 Host 程序并启动 Kernel。程序遇到 ACL 错误时，会打印出错的调用并释放已申请的资源。

先编译负责启动计算的 Host 程序。`-I` 指定头文件目录，`-L` 指定库目录，`-lascendcl` 和 `-lacl_rtc` 链接运行时所需的库。


In [ ]:
%%bash
set -e
mkdir -p build
g++ -std=c++17 src/rope_simd_rtc.cpp \
  -I"$ASCEND_HOME_PATH/include" -I"$ASCEND_HOME_PATH/include/acl" \
  -L"$ASCEND_HOME_PATH/lib64" -Wl,-rpath,"$ASCEND_HOME_PATH/lib64" \
  -lascendcl -lacl_rtc -o build/rope_simd_rtc


运行时，Host 程序读取并编译 `src/rope_simd_kernel.cpp`，再提交到 NPU。修改 Kernel 后，重新运行下面的单元格即可。


In [ ]:
%%bash
# 学生修改：--warmup 是预热次数，--repeat 是统计次数。
./build/rope_simd_rtc --kernel src/rope_simd_kernel.cpp --warmup 2 --repeat 5


成功时输出 `ROPE_RESULT status=PASS cases=4/4`。重点查看 `max_error`、`tolerance` 和 `device_mean_us`。

`device_mean_us` 包含 Kernel 提交和等待完成，不含 RTC 编译、H2D、D2H。


### SIMT 模板

`rope_simt_950.asc` 用 `blockIdx/threadIdx` 计算全局线程号，一个线程处理一个 pair，并用边界判断防止越界：

```cpp
unsigned int p = blockIdx.x * blockDim.x + threadIdx.x;
if (p >= count) {
    return;
}
out_even[p] = x_even[p] * cos[p] - x_odd[p] * sin[p];
out_odd[p]  = x_even[p] * sin[p] + x_odd[p] * cos[p];
```

模板面向 Ascend 950（`dav-3510`、`--enable-simt`），使用与 SIMD 相同的公式和数据布局。本课程 A3 环境只运行 SIMD，SIMT 部分阅读源码即可。


### 课后实践

解释 SIMD 中“向量指令一次处理一段数据”和 SIMT 中“一个线程处理一个 pair”的区别。两种写法为什么要保持相同的输入输出布局？


完成本节练习后查看：[参考答案](answer/06.06_answer.md)